# Pota Takip - YOLO11l Eğitimi (Colab, Doğruluk Öncelikli)

Bu notebook Google Drive kullanmaz — `dataset.zip` dosyasını (`<proje-dizini>`, ~444MB) doğrudan tarayıcınızdan Colab'a yüklersiniz (aşağıdaki 3. hücre).

Doğruluk önceliğiyle **YOLO11l** (large) modeli, **imgsz=960** (kaynak görüntüler 2560x1440 olduğu için detay kaybını azaltmak amacıyla) ve batch'i GPU belleğine göre otomatik ayarlayacak şekilde yapılandırıldı. Bu, nano modele göre daha uzun sürer (T4'te muhtemelen 1.5-3 saat, 587 görsellik veri setinde).

Önce **Runtime > Change runtime type > GPU (T4)** seçin, sonra hücreleri sırayla çalıştırın.

In [ ]:
!pip install -q ultralytics
import torch
print('GPU var mı:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## Dataset'i doğrudan yükleme
Aşağıdaki hücreyi çalıştırınca bir "Dosya Seç" penceresi açılacak. Bilgisayarınızdan `dataset.zip` dosyasını seçin. Yükleme tarayıcıdan doğrudan Colab'a gider, Drive'a hiç uğramaz. Bağlantınız yavaşsa birkaç dakika sürebilir, sekmeyi kapatmayın.

In [ ]:
from google.colab import files

uploaded = files.upload()  # burada dataset.zip'i seçin
print(list(uploaded.keys()))

In [ ]:
!mkdir -p /content/work
!unzip -q dataset.zip -d /content/work
!ls /content/work/dataset

In [ ]:
# data.yaml içindeki path'i Colab ortamına göre yeniden yaz
data_yaml = '''# Pota Takip Projesi - YOLO11 Konfigürasyonu
path: /content/work/dataset
train: train/images
val: val/images

nc: 1
names:
  0: pota
'''
with open('/content/work/dataset/data.yaml', 'w') as f:
    f.write(data_yaml)
print(data_yaml)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')  # COCO on-egitimli hafif model

results = model.train(
    data='/content/work/dataset/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    patience=20,
    project='/content/runs',
    name='pota_yolo11n',
)

In [ ]:
# Validasyon metrikleri (mAP, precision, recall)
metrics = model.val()
print(metrics)

## Eğitilen modeli doğrudan indirme
Drive'a değil, doğrudan bilgisayarınıza indirir.

In [ ]:
from google.colab import files

files.download('/content/runs/pota_yolo11n/weights/best.pt')

## Sırada ne var
- İndirdiğiniz `best.pt` dosyasını yerel makinede test edebilirsiniz.
- Bir sonraki adım: bu modeli `model.track(source=..., tracker='bytetrack.yaml')` ile kamera akışına uygulayıp giriş/çıkış ve süre takibini kurmak.